
# 04 — End-to-End Agent Authentication & Authorization

## Objective

Understand how an enterprise AI agent securely handles an employee's request—from the HTTP API boundary to retrieving authorized business data.

**Core principle:** The LLM decides *what information to request*, but trusted application code determines *who the user is and what they are allowed to access*.

## End-to-End Security Flow

```text
Employee sends HTTP request
  Authorization: Bearer <JWT>
  Body: {"claim_id": "C-1042"}
           |
           v
1. FastAPI — Authentication
   Extract bearer token using Depends()
   Verify signature, issuer, audience, expiration
   Invalid token → HTTP 401
           |
           v
2. Employee Directory — Application Eligibility
   Resolve employee from verified token subject
   Check employee exists and is active
   Retrieve tenant from trusted directory
   Unknown/disabled employee → HTTP 403
           |
           v
3. Construct Trusted Principal
   Principal(employee_id, tenant_id)
   Identity comes from trusted application code,
   NEVER from the LLM or request body
           |
           v
4. LangGraph — Trusted Context Injection
   Inject Principal through InvestigationContext
   LLM-visible tool arguments: claim_id only
   Identity is hidden from model-controlled arguments
           |
           v
5. Authorization-Aware Tool Gateway
   Verify tenant isolation
   Verify resource-level access grant
   Unauthorized claim → HTTP 403
           |
           v
6. Retrieve & Return Authorized Evidence
   Read permitted claim data
   Return HTTP 200 with claim details
```

## Four Distinct Security Responsibilities

| Layer | Question answered | Enforcement |
|---|---|---|
| Authentication | Who is calling? | JWT signature and claims verification |
| Application eligibility | Is this employee currently allowed to use the application? | Trusted employee directory |
| Identity propagation | Can the LLM impersonate another employee? | Server-owned Principal injected through LangGraph runtime |
| Resource authorization | Can this employee access this particular claim? | Tenant isolation and explicit resource grants |

## Expected Security Behavior

| Scenario | HTTP result |
|---|---|
| Alice accesses authorized claim | 200 OK |
| Bob accesses claim without permission | 403 Forbidden |
| Forged, expired, or wrong-audience JWT | 401 Unauthorized |
| Unknown or disabled employee | 403 Forbidden |
| Missing authentication token | 401 Unauthorized |
| Caller attempts to override identity in request body | 422 Validation Error |

## Key Architectural Lessons

1. **Authentication is not authorization.** A valid JWT does not automatically grant access to a claim.
2. **FastAPI Depends() does not validate tokens automatically.** The authentication dependency must explicitly verify the JWT.
3. **A valid JWT does not guarantee current eligibility.** An employee may be disabled after token issuance.
4. **Never trust model-generated identity.** The application constructs the Principal from verified identity and trusted employee records.
5. **ToolRuntime injects context; it does not authenticate users.** Authentication must happen before invoking the graph.
6. **Authorization happens before returning data.** The gateway enforces tenant and resource permissions independently of the model.
7. **Fail closed.** Missing identity, invalid credentials, unavailable authorization information, or missing grants must not result in access.
8. **Tests must verify both success and rejection.** Legitimate investigations should work while unauthorized requests are denied.

## Scope and Limitations

This notebook uses synthetic JWTs, an in-memory signing key, CSV-based employee records, and a simulated LangGraph tool call.

It demonstrates the security boundary without requiring a running server or an LLM.

It does **not** yet implement production identity federation, live employee-status synchronization, field-level authorization, or a hardened audit system.

**Important:** Employee records are loaded when the application starts. Changes to the CSV are not automatically reflected in a running instance. Freshness and revocation handling will be addressed separately.

## Mental Model

**Authenticate → Verify eligibility → Establish trusted identity → Inject context → Authorize resource → Retrieve evidence**

The agent may reason about the investigation, but it must never become the authority for identity or access control.


## Initialize the app

In [13]:
import secrets
from pathlib import Path

import jwt
from fastapi.testclient import TestClient

from investigation_agent_security_lab.api import create_app
from investigation_agent_security_lab.identity import (
    LAB_ISSUER,
    LAB_AUDIENCE,
)
from datetime import datetime, timedelta, timezone

cwd = Path.cwd()
ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

LAB_KEY = secrets.token_bytes(32)
app = create_app(ROOT, LAB_KEY)
client = TestClient(app)

print("Lab API initialized")

Lab API initialized


## Create a synthetic signed token

In [14]:
def create_test_token(
    employee_id: str,
    key: bytes = LAB_KEY,
    audience: str = LAB_AUDIENCE,
    expires_in: int = 900,
):
    now = datetime.now(timezone.utc)
    return jwt.encode(
        {
            "sub": employee_id,
            "iss": LAB_ISSUER,
            "aud": audience,
            "iat": now,
            "exp": now + timedelta(seconds=expires_in),
        },
        key,
        algorithm="HS256",
    )


def investigate(token: str, claim_id: str = "C-1042"):
    return client.post(
        "/investigate",
        json={"claim_id": claim_id},
        headers={"Authorization": f"Bearer {token}"},
    )

### Alice investigates her authorized claim

In [15]:
alice_token = create_test_token("E-101")
response = investigate(alice_token)

print(response.status_code)
print(response.json())

assert response.status_code == 200
assert response.json()["claim"]["tenant_id"] == "northstar"

200
{'claim': {'tenant_id': 'northstar', 'claim_id': 'C-1042', 'customer_id': 'CUS-100', 'status': 'open', 'loss_type': 'water damage', 'amount_usd': '4200'}}


### Bob attempts the same claim

In [16]:
bob_token = create_test_token("E-102")
response = investigate(bob_token)

print(response.status_code, response.json())

assert response.status_code == 403

403 {'detail': 'Access denied'}


### Try impersonation with an invalid signing key

In [17]:
attacker_token = create_test_token(
    "E-101",
    key=secrets.token_bytes(32),
)

response = investigate(attacker_token)

print(response.status_code, response.json())

assert response.status_code == 401

401 {'detail': 'Invalid credentials'}


### Try supplying a forged identity in the request body

In [18]:
response = client.post(
    "/investigate",
    json={
        "claim_id": "C-1042",
        "employee_id": "E-201",
        "tenant_id": "harbor",
    },
    headers={"Authorization": f"Bearer {alice_token}"},
)

print(response.status_code, response.json())

assert response.status_code == 422

422 {'detail': [{'type': 'extra_forbidden', 'loc': ['body', 'employee_id'], 'msg': 'Extra inputs are not permitted', 'input': 'E-201'}, {'type': 'extra_forbidden', 'loc': ['body', 'tenant_id'], 'msg': 'Extra inputs are not permitted', 'input': 'harbor'}]}


## Disabled-employee experiment

In [19]:
import shutil
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as temp:
    temp_root = Path(temp)

    synthetic = temp_root / "data" / "synthetic"
    synthetic.mkdir(parents=True)

    for filename in [
        "claims.csv",
        "access_grants.csv",
        "employees.csv",
    ]:
        shutil.copy(
            ROOT / "data/synthetic" / filename,
            synthetic / filename,
        )

    app = create_app(temp_root, LAB_KEY)
    test_client = TestClient(app)

    headers = {
        "Authorization": f"Bearer {alice_token}"
    }

    # Alice is initially active.
    before = test_client.post(
        "/investigate",
        json={"claim_id": "C-1042"},
        headers=headers,
    )

    print("Before disabling:", before.status_code)

    # Disable Alice without rebuilding the application.
    employee_file = synthetic / "employees.csv"
    contents = employee_file.read_text()
    employee_file.write_text(
        contents.replace(
            "E-101,northstar,adjuster,true",
            "E-101,northstar,adjuster,false",
        )
    )

    after = test_client.post(
        "/investigate",
        json={"claim_id": "C-1042"},
        headers=headers,
    )

    print("After disabling:", after.status_code)

    assert before.status_code == 200
    assert after.status_code == 403

Before disabling: 200
After disabling: 200


AssertionError: 

In [10]:
alice_token = create_test_token("E-101")
response = investigate(alice_token)

print(response.status_code)
print(response.json())

assert response.status_code == 200
assert response.json()["claim"]["tenant_id"] == "northstar"

200
{'claim': {'tenant_id': 'northstar', 'claim_id': 'C-1042', 'customer_id': 'CUS-100', 'status': 'open', 'loss_type': 'water damage', 'amount_usd': '4200'}}
